# F1 · Paquete final + gate

**Spec:** [`docs/spec_F1_codex_final_products.md`](../docs/spec_F1_codex_final_products.md)  |  **Bloque:** F · Paquete  |  **Run de este set:** `ROXs42Bb_realigned`

Consolida A→E en el paquete final y aplica el gate (semáforo por etapa + limitaciones aceptadas).

| | |
|---|---|
| **Entrada** | Todos los QC A→E |
| **Salida (QC/productos)** | `report/run_summary.json`, `report/report.md`, figuras/tablas |
| **Consume aguas abajo** | Revisión humana / decisión de publicación |


## Qué hace F1 y cómo gatea

F1 consolida todos los QC de A→E en el **paquete final** y aplica el **gate**: un semáforo por etapa (green/yellow/red) + una **política de gate congelada** que degrada ciertos rojos *documentados* a yellow **'limitación aceptada'** (no los esconde: los conserva y anota). **Cualquier OTRO rojo bloquea.**

**Semáforo:** 15 yellow, 1 green (B1), 1 `not_run` (A3), **0 rojos** → overall **yellow**.

**4 limitaciones aceptadas** (congeladas, hash `e4478990`) — las mismas que fuimos viendo etapa por etapa:
- **A4/M5 STAT**: la varianza STAT subestima ~4–6× por la covarianza del remuestreo (inherente al drizzle).
- **D2/v3 continuo**: sistemático de NIVEL inter-método en el rojo (residuo de halo cromático; ya lo referenciamos a controles).
- **E2/T2**: el test de forma-PSF reinterpretado para una NO-detección (el máximo no es PSF → apoya la no-detección).
- **E4/v4_hierarchy**: la patología de borde del throughput (aperture/ls insensibles; canónico psffit no afectado).

`hash_chain` = **pass** (proveniencia de productos trazable). **F1 se niega correctamente a dar luz verde de paper** mientras el A-block siga provisional: es **yellow, no green**.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python scripts/build_report.py --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('report/run_summary.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python scripts/build_report.py --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('report/run_summary.json', RUN_ID)
nb.show(qc, keys=['overall_status', 'accepted_limitations_hash', 'schema_version'], title='F1')


## Resultados que llevaron a la conclusión

Semáforo por etapa, las 4 limitaciones aceptadas y la cadena de hash del `report/run_summary.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('F1', 'report/run_summary.json'):
        q = nb.load_qc('report/run_summary.json', RUN_ID)
        from collections import Counter
        cnt = Counter(s['status'] for s in q['stages'])
        print('overall_status:', q['overall_status'], '|', dict(cnt))
        hc = q.get('hash_chain', {})
        print(f"hash_chain: {hc.get('status')} ({len(hc.get('checks', []))} checks) | "
              f"open_issues: {len(q['open_issues'])} | gate hash: {q['gate_policy']['accepted_limitations_hash']}")
        print('\nsemáforo por etapa:')
        for s in q['stages']:
            mark = f"  (limitación aceptada ×{s['accepted_limitations']})" if s['accepted_limitations'] else ''
            print(f"   {s['stage']:20s} {s['status']:9s} issues={s['issue_count']}{mark}")
        print('\n4 limitaciones aceptadas:')
        for a in q['accepted_limitations']:
            print(f"   {a['path']:32s} {a['reason'][:75]}...")


## Plot 1 — el semáforo del gate

Status de las 17 etapas. **0 rojos** → overall **yellow**. B1 verde, A3 `not_run`, el resto yellow; `⚠×1` marca las 4 etapas con una limitación aceptada (A4, D2, E2, E4).


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('report/run_summary.json', RUN_ID)
    st = q['stages']
    col = {'green': 'tab:green', 'yellow': 'gold', 'red': 'tab:red', 'not_run': '0.8'}
    names = [s['stage'] for s in st]; stats = [s['status'] for s in st]; accl = [s['accepted_limitations'] for s in st]
    from collections import Counter
    cnt = Counter(stats)
    resumen = ', '.join(f'{v} {k}' for k, v in sorted(cnt.items(), key=lambda kv: -kv[1]))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(range(len(names)), [1] * len(names), color=[col.get(s, '0.5') for s in stats])
    for i, (n, s, a) in enumerate(zip(names, stats, accl)):
        ax.text(0.02, i, f"{n}  [{s}]" + (f'  ⚠×{a}' if a else ''), va='center', fontsize=8, color='k')
    ax.set_yticks([]); ax.set_xticks([]); ax.invert_yaxis(); ax.set_xlim(0, 1)
    ax.set_title(f"F1 · semáforo: overall={q['overall_status'].upper()} "
                 f"({cnt.get('red', 0)} rojos; {resumen})")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'f1_report'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'gate.png', dpi=110); print('figura ->', outdir / 'gate.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — las 4 limitaciones aceptadas

Los 4 rojos degradados a yellow (política de gate congelada). Cada uno es inherente, reinterpretado o documentado — y ya los revisamos en A4, D2, E2, E4.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('report/run_summary.json', RUN_ID)
    al = q['accepted_limitations']
    labels = {'m5_stat.status': 'A4/M5 STAT', 'checks.v3_continuum_stable.ok': 'D2/v3 continuo rojo',
              't2.status': 'E2/T2 forma-PSF', 'checks.v4_hierarchy.status': 'E4 jerarquía'}
    fig, ax = plt.subplots(figsize=(11, 3.2))
    for i, a in enumerate(al):
        lab = labels.get(a['path'], a['path'])
        ax.text(0.01, len(al) - 1 - i, f'● {lab}', fontsize=10, weight='bold', va='center')
        ax.text(0.22, len(al) - 1 - i, a['reason'][:110] + '...', fontsize=8, va='center')
    ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(al) - 0.5); ax.axis('off')
    ax.set_title(f"F1 · {len(al)} limitaciones aceptadas (rojos degradados; hash {q['gate_policy']['accepted_limitations_hash']})")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'f1_report'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'accepted_limitations.png', dpi=110); print('figura ->', outdir / 'accepted_limitations.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Gate `overall_status: yellow`, 0 rojos bloqueantes** en el realineado (era rojo en ADP): 15 yellow, 1 green (B1), 1 not_run (A3).
- Política de gate **congelada** (hash e4478990): SOLO 4 rojos específicos (A4/M5, D2/v3, E2/T2, E4/hierarchy) bajan a 'limitación aceptada'; cualquier OTRO rojo bloquea. · [`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)
- Las limitaciones aceptadas se **conservan y anotan** (no se esconden); `hash_chain` pass (proveniencia trazable).
- F1 se niega correctamente a dar **luz verde de paper** mientras el A-block siga provisional (yellow, no green).


## Conclusión (registrada)

**F1: paquete final; overall `yellow`, 0 rojos bloqueantes.**

- **Fecha:** consolidado 2026-07-09/10 sobre el run realineado.
- **Semáforo:** 15 yellow, 1 green (B1), 1 not_run (A3), 0 rojos.
- **4 limitaciones aceptadas** (congeladas, documentadas, no ocultas): A4/M5, D2/v3, E2/T2, E4/hierarchy.
- **hash_chain pass**; 29 open_issues agregados.
- **Yellow (no green):** F1 se niega a dar luz verde de paper mientras el A-block siga provisional — es el comportamiento correcto.
- **Endpoint del paquete:** no-detección de Hα → Ṁ ≲ 8×10⁻¹³ M☉/yr, compañero real ligado (caracterización en el bloque G).
